# 🧑‍⚖️ Avatar Realista — Neto de Moraes Associados
Pipeline rapido (T4): **foto** + **audio de referencia** -> **video do avatar falando**.

1. Runtime -> Change runtime type -> **T4 GPU** -> Save
2. **Run all**
3. Na celula 3, envie 1 foto (rosto nitido, olhando para a camera) e 1 audio (10-60s da sua voz)
4. O video sai no final — baixe na ultima celula

Aviso: roda na memoria da sessao do Colab. Se desconectar, rode tudo de novo.

In [ ]:
# 1) Instalacao (~5 min na primeira vez)
import os
os.environ["COQUI_TOS_AGREED"] = "1"  # aceita licenca do XTTS automaticamente
!pip -q install coqui-tts gradio==4.44.1 huggingface_hub
!git clone -q https://github.com/OpenTalker/SadTalker /content/SadTalker
%cd /content/SadTalker
!pip -q install face_alignment==1.3.5 imageio==2.19.3 imageio-ffmpeg==0.4.7 librosa==0.9.2 resampy==0.3.1 pydub==0.25.1 scipy==1.10.1 kornia==0.6.8 yacs==0.1.8 joblib==1.1.0 scikit-image==0.19.3 basicsr==1.4.2 facexlib==0.3.0 gfpgan av safetensors
# --- patches de compatibilidade (numpy 2.x / torchvision novo) ---
!sed -i 's/np.VisibleDeprecationWarning/Warning/g' src/face3d/util/preprocess.py
!grep -rl "np\.float\b" src | xargs -r sed -i 's/np\.float\b/float/g'
!grep -rl "np\.int\b" src | xargs -r sed -i 's/np\.int\b/int/g'
!grep -rl "np\.bool\b" src | xargs -r sed -i 's/np\.bool\b/bool/g'
import glob as _g
for _p in _g.glob('/usr/local/lib/python3*/dist-packages/basicsr/data/degradations.py') + _g.glob('/usr/local/lib/python3*/site-packages/basicsr/data/degradations.py'):
    _s = open(_p).read()
    if 'functional_tensor' in _s:
        open(_p, 'w').write(_s.replace('functional_tensor import rgb_to_grayscale', 'functional import rgb_to_grayscale'))
        print('basicsr corrigido:', _p)
!bash scripts/download_models.sh
# --- verificacao das dependencias criticas ---
import importlib
_faltando = []
for _m in ['kornia','facexlib','basicsr','gfpgan','face_alignment','librosa','imageio','pydub','yacs','safetensors','av']:
    try:
        importlib.import_module(_m)
    except Exception as _e:
        print('FALHOU', _m, '->', _e)
        _faltando.append(_m)
print("OK: instalacao concluida" if not _faltando else f"ATENCAO: faltam {_faltando} - rode a celula de instalacao robusta")

In [ ]:
# 2) Envie as FOTOS (pode ser mais de uma) e depois o AUDIO
from google.colab import files
import os, shutil

os.makedirs('/content/input', exist_ok=True)
!rm -f /content/input/*

print("PASSO 1 - Envie as FOTOS (pode selecionar varias de uma vez):")
up1 = files.upload()
photos = []
for f in up1:
    dst = '/content/input/' + f
    shutil.move(f, dst)
    if os.path.splitext(f)[1].lower() in ('.jpg','.jpeg','.png'):
        photos.append(dst)
if not photos:
    raise ValueError("Nenhuma foto enviada. Rode esta celula de novo.")
# usa a maior foto como principal (maior resolucao = melhor resultado)
photo = max(photos, key=os.path.getsize)
print(f"{len(photos)} foto(s) recebida(s). Principal: {photo}")

print("\nPASSO 2 - Agora envie o AUDIO da sua voz (10-60s, .wav/.mp3/.m4a):")
up2 = files.upload()
audio = None
for f in up2:
    dst = '/content/input/' + f
    shutil.move(f, dst)
    if os.path.splitext(f)[1].lower() in ('.wav','.mp3','.m4a','.ogg','.flac'):
        audio = dst
if audio is None:
    raise ValueError("Nenhum audio enviado. Rode esta celula de novo e envie o arquivo de voz.")
print('Foto principal:', photo, '| Fotos extras p/ fase 2:', len(photos)-1, '| Audio:', audio)

In [ ]:
# 3) Animacao labial: foto + audio -> video (SadTalker)
!python inference.py --driven_audio "{audio}" --source_image "{photo}" \
  --result_dir /content/output --still --preprocess full --enhancer gfpgan
import glob
video = sorted(glob.glob('/content/output/**/*.mp4', recursive=True))[-1]
print('Video gerado:', video)

In [ ]:
# 4) (Opcional) Clonar voz: gere um novo audio com a sua voz (XTTS-v2)
texto = "Seu plano de saude negou um tratamento? A gente reverte. Fale com a Neto de Moraes Associados."
!pip -q install coqui-tts
import os
os.environ["COQUI_TOS_AGREED"]="1"
from TTS.api import TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
tts.tts_to_file(text=texto, speaker_wav=audio, language="pt-br",
                file_path="/content/input/voz-clonada.wav")
print('Audio clonado: /content/input/voz-clonada.wav')
# Para usar: rode a celula 3 trocando audio por '/content/input/voz-clonada.wav'

In [ ]:
# 5) Download do video
from google.colab import files
files.download(video)